# Notebook 05: Interpretability Analysis

## HIV Drug Resistance Prediction with ESM-2

---

**Objective**: Validate biological relevance of ESM-2 predictions.

**Analyses**:
1. DRM enrichment in high-attention positions
2. Novel position discovery
3. SHAP feature importance

**Targets**:
- DRM Enrichment: 2.48x
- Novel positions: 228

---

In [ ]:
import os
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm import tqdm

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_processing import load_unified_data, get_drug_list
from feature_engineering import load_esm2_model, extract_attention_weights
from interpretability import (
    load_known_drms, get_drug_specific_drms,
    compute_drm_enrichment, find_novel_positions,
    compute_attention_differential, summarize_drm_validation
)
from visualization import plot_attention_heatmap, plot_drm_enrichment

# Random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Imports complete")

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Data and Model

In [ ]:
# Load unified data
unified_data = load_unified_data(DATA_DIR)

print("Loaded data:")
for dc, data in unified_data.items():
    print(f"  {dc}: {len(data['sequences'])} sequences")

In [ ]:
# Load ESM-2 model
print("\nLoading ESM-2 model...")
model, alphabet, batch_converter, device = load_esm2_model("esm2_t33_650M_UR50D")
print(f"Model loaded on: {device}")

## 2. Known DRM Reference Data

Drug Resistance Mutations from IAS-USA 2022 guidelines.

In [ ]:
# Load known DRMs
print("Known DRM positions (IAS-USA 2022):")
print("-" * 50)

for drug_class in ['PI', 'NRTI', 'NNRTI']:
    drm_positions = load_known_drms(drug_class)
    print(f"\n{drug_class}: {len(drm_positions)} positions")
    print(f"  Positions: {sorted(drm_positions)[:15]}...")

## 3. Extract Attention for Resistant vs Susceptible

In [ ]:
# Configuration
MAX_SAMPLES_PER_CLASS = 100  # For efficiency

attention_results = {}

In [ ]:
# Extract attention differential for each drug
for drug_class, data in unified_data.items():
    print(f"\n{'='*60}")
    print(f"PROCESSING {drug_class}")
    print(f"{'='*60}")
    
    attention_results[drug_class] = {}
    
    sequences = data['sequences']
    phenotypes = data['phenotypes']
    drugs = get_drug_list(drug_class)
    
    for drug in tqdm(drugs, desc=f"{drug_class} drugs"):
        # Get labels
        label_col = f"{drug}_class2"
        if label_col not in phenotypes.columns:
            continue
        
        labels = phenotypes[label_col].values
        valid_mask = ~np.isnan(labels)
        
        valid_seqs = [s for s, v in zip(sequences, valid_mask) if v]
        valid_labels = labels[valid_mask].astype(int)
        
        # Compute attention differential
        result = compute_attention_differential(
            valid_seqs, valid_labels,
            model, alphabet, device,
            max_samples=MAX_SAMPLES_PER_CLASS,
            random_state=RANDOM_SEED
        )
        
        if result:
            attention_results[drug_class][drug] = result
            print(f"  {drug}: R={result['n_resistant']}, S={result['n_susceptible']}")

## 4. DRM Enrichment Analysis

In [ ]:
# Compute DRM enrichment for all drugs
validation_results = []

print("\nDRM Validation Analysis:")
print("-" * 60)

for drug_class in attention_results.keys():
    print(f"\n{drug_class}:")
    
    class_drm = load_known_drms(drug_class)
    
    for drug, result in attention_results[drug_class].items():
        differential = result['differential']
        
        # Get drug-specific DRMs
        drm_positions = get_drug_specific_drms(drug, drug_class)
        
        # Compute enrichment at different top-k
        for top_k in [10, 20, 30]:
            enrichment = compute_drm_enrichment(differential, drm_positions, top_k=top_k)
            
            validation_results.append({
                'drug_class': drug_class,
                'drug': drug,
                'top_k': top_k,
                'enrichment_ratio': enrichment['enrichment_ratio'],
                'p_value': enrichment['p_value'],
                'observed': enrichment['observed'],
                'expected': enrichment['expected']
            })
        
        # Report top-20 enrichment
        top20 = [r for r in validation_results if r['drug'] == drug and r['top_k'] == 20][-1]
        sig = "*" if top20['p_value'] < 0.05 else ""
        print(f"  {drug}: {top20['enrichment_ratio']:.2f}x enrichment{sig}")

validation_df = pd.DataFrame(validation_results)

In [ ]:
# Summary statistics
summary = summarize_drm_validation(validation_results, top_k=20)

print("\n" + "="*60)
print("DRM ENRICHMENT SUMMARY (top-20)")
print("="*60)
print(f"\nMean enrichment: {summary['mean_enrichment']:.2f}x")
print(f"Median enrichment: {summary['median_enrichment']:.2f}x")
print(f"Significant (p<0.05): {summary['n_significant']}/{summary['n_drugs']} ({summary['pct_significant']:.1f}%)")

## 5. Novel Position Discovery

In [ ]:
# Find novel positions (high attention, not in DRM list)
novel_positions = []

print("\nNovel Position Discovery:")
print("-" * 60)

for drug_class in attention_results.keys():
    print(f"\n{drug_class}:")
    
    class_drm = load_known_drms(drug_class)
    
    for drug, result in attention_results[drug_class].items():
        differential = result['differential']
        
        novel = find_novel_positions(differential, class_drm, top_k=30)
        
        for pos_info in novel:
            novel_positions.append({
                'drug_class': drug_class,
                'drug': drug,
                **pos_info
            })
        
        print(f"  {drug}: {len(novel)} novel positions in top-30")

novel_df = pd.DataFrame(novel_positions)

In [ ]:
# Most frequent novel positions
if len(novel_df) > 0:
    print("\nMost Frequent Novel Positions:")
    print("-" * 50)
    
    for drug_class in novel_df['drug_class'].unique():
        dc_novel = novel_df[novel_df['drug_class'] == drug_class]
        position_counts = dc_novel['position'].value_counts().head(5)
        
        print(f"\n{drug_class}:")
        for pos, count in position_counts.items():
            print(f"  Position {pos}: appears in {count} drug(s)")
    
    print(f"\nTotal novel positions: {novel_df['position'].nunique()}")

## 6. Visualization

In [ ]:
# DRM enrichment visualization
if len(validation_df) > 0:
    fig = plot_drm_enrichment(
        validation_df,
        save_path=FIGURES_DIR / 'drm_enrichment.png'
    )
    plt.show()

In [ ]:
# Attention profile for representative drugs
representative_drugs = {
    'PI': 'DRV',
    'NRTI': 'TDF',
    'NNRTI': 'EFV'
}

for drug_class, drug in representative_drugs.items():
    if drug_class in attention_results and drug in attention_results[drug_class]:
        result = attention_results[drug_class][drug]
        drm_positions = get_drug_specific_drms(drug, drug_class)
        
        seq_len = 99 if drug_class == 'PI' else 240
        
        fig = plot_attention_heatmap(
            result['differential'],
            drm_positions,
            seq_len,
            drug, drug_class,
            save_path=FIGURES_DIR / f'attention_profile_{drug}.png'
        )
        plt.show()

## 7. Save Results

In [ ]:
# Save validation results
validation_df.to_csv(RESULTS_DIR / 'drm_validation_results.csv', index=False)
print(f"Saved: drm_validation_results.csv")

# Save novel positions
novel_df.to_csv(RESULTS_DIR / 'novel_positions.csv', index=False)
print(f"Saved: novel_positions.csv")

# Save attention results
with open(RESULTS_DIR / 'attention_results.pkl', 'wb') as f:
    pickle.dump(attention_results, f)
print(f"Saved: attention_results.pkl")

## Summary

**Expected results:**
- DRM enrichment: ~2.48x in top-20 attention positions
- Novel positions: ~228 unique positions across all drugs
- Significant enrichment in majority of drugs (p<0.05)

**Key findings:**
- ESM-2 attention weights align with known resistance biology
- Novel positions warrant experimental validation
- Supports hypothesis that PLMs capture resistance-relevant features

**Next steps:**
- Notebook 06: External validation and calibration